# Per-Day Sampled Scatter Decoding Orchestrator — Quality-Filtered Speech

Identical to `scat_classifier_sampled_nocv_filtered_speech.ipynb` but restricted to
words spoken within a single calendar day during **active hours** (default 09:00–23:00 local time).

One SLURM job is submitted per *(patient, date, resample)* triplet.  
Outputs land in `{vad_root}/{patient}/standard_decoding_analysis/{BASE_RUN_NAME}/{date}/`.

**`REUSE_PATIENT_CONSENSUS = True`** (default): borrows hyperparameters from the already-finished
patient-level run (`scat_xgboost_sampled_norm_filtered`) and skips the 3-resample hyper-search per day,
going straight to the 20-resample fixed-params stage.  
Set to `False` to run the full two-phase search independently for each day.

### 1. Imports

In [1]:
import json
import subprocess
from pathlib import Path

import dill as pickle
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

### 2. Configuration

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
PATIENTS    = ['YEY', 'YEZ', 'YFA', 'YFB', 'YFC', 'YFD', 'YFE', 'YFF', 'YFG',
               'YFI', 'YFK', 'YFL', 'YFM', 'YFN', 'YFP', 'YFQ', 'YFR', 'YFT', 'YFU', 'YFV']

VAD_ROOT    = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new')
LEGACY_ROOT = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_out')

BASE_RUN_NAME          = 'scat_xgboost_sampled_norm_filtered_per_day'
PATIENT_CONSENSUS_RUN  = 'scat_xgboost_sampled_norm_filtered'   # source of borrowed params

CLUSTER_PREDS_OVERRIDE = {}
FRS_PATH_OVERRIDE      = {}

# ── Day-partition settings ────────────────────────────────────────────────────
LOCAL_TZ           = 'America/Chicago'
ACTIVE_HOURS_START = 9
ACTIVE_HOURS_END   = 23
MIN_WORDS_PER_DAY  = 200

# ── Two-phase strategy ────────────────────────────────────────────────────────
# True  → skip per-day hyperparameter search; borrow consensus from the patient-level run
# False → run full 3+17 pipeline independently per day
REUSE_PATIENT_CONSENSUS = False

N_RESAMPLES  = 20
FIRST_STAGE  = [0, 1, 2]
SECOND_STAGE = list(range(3, N_RESAMPLES))
SEED_STRIDE  = 42

PYTHON_BIN  = '/scratch/tahaismail424/miniforge3/envs/speech_247_env/bin/python3'
WORKER_PATH = Path('/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!'
                   '/standard_decoding_analysis/scat_classifier_worker.py')

PARTITION = 'Universal'
QOS       = 'big_batch_tier'
CPUS      = 8
MEM       = '40G'
TIME      = '48:00:00'

TEST_SIZE    = 0.2
SEARCH_ITERS = 40
CV_SPLITS    = 4
N_SHUFFLES   = 50
SCORING      = 'f1_macro'
PERM_METRIC  = 'f1_macro'

print('patients:', PATIENTS)
print('worker:', WORKER_PATH)
print(f'active hours: {ACTIVE_HOURS_START}:00 – {ACTIVE_HOURS_END}:00 {LOCAL_TZ}')
print(f'reuse patient consensus: {REUSE_PATIENT_CONSENSUS}')

patients: ['YEY', 'YEZ', 'YFA', 'YFB', 'YFC', 'YFD', 'YFE', 'YFF', 'YFG', 'YFI', 'YFK', 'YFL', 'YFM', 'YFN', 'YFP', 'YFQ', 'YFR', 'YFT', 'YFU', 'YFV']
worker: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/standard_decoding_analysis/scat_classifier_worker.py
active hours: 9:00 – 23:00 America/Chicago
reuse patient consensus: False


### 3. Compute & Save Per-Day Word Indices Per Patient

In [3]:
def compute_day_indices(patient: str) -> 'dict[str, Path]':
    patient_root = VAD_ROOT / patient
    filtered_csv = patient_root / f'{patient}_word_df_filtered.csv'

    if not filtered_csv.exists():
        print(f'  [{patient}] missing {filtered_csv.name}')
        return {}

    df = pd.read_csv(filtered_csv, usecols=['original_word_idx', 'utc_word_start'])
    df['utc_word_start'] = pd.to_datetime(df['utc_word_start'], utc=True)
    df['local_dt'] = df['utc_word_start'].dt.tz_convert(LOCAL_TZ)
    df['hour']     = df['local_dt'].dt.hour
    df['date_str'] = df['local_dt'].dt.strftime('%Y-%m-%d')

    active = df[(df['hour'] >= ACTIVE_HOURS_START) & (df['hour'] < ACTIVE_HOURS_END)]

    day_paths = {}
    for date_str, grp in active.groupby('date_str'):
        if len(grp) < MIN_WORDS_PER_DAY:
            print(f'  [{patient}] {date_str}: only {len(grp)} words — skipping')
            continue
        idx_dir  = patient_root / 'standard_decoding_analysis' / BASE_RUN_NAME / date_str
        idx_dir.mkdir(parents=True, exist_ok=True)
        idx_path = idx_dir / f'{patient}_{date_str}_word_idx.npy'
        word_idx = grp['original_word_idx'].values.astype(np.int64)
        np.save(idx_path, word_idx)
        day_paths[date_str] = idx_path
        print(f'  [{patient}] {date_str}: {len(word_idx):,} words → {idx_path.name}')

    return day_paths


print(f'Computing per-day word indices for {len(PATIENTS)} patients...\n')
patient_day_idx_paths = {}
for patient in PATIENTS:
    day_paths = compute_day_indices(patient)
    if day_paths:
        patient_day_idx_paths[patient] = day_paths

total_pairs = sum(len(v) for v in patient_day_idx_paths.values())
print(f'\nTotal (patient, day) pairs ready: {total_pairs}')

Computing per-day word indices for 20 patients...

  [YEY] 2024-04-01: 16,510 words → YEY_2024-04-01_word_idx.npy
  [YEY] 2024-04-02: 23,841 words → YEY_2024-04-02_word_idx.npy
  [YEZ] 2024-04-09: 6,698 words → YEZ_2024-04-09_word_idx.npy
  [YEZ] 2024-04-10: 16,464 words → YEZ_2024-04-10_word_idx.npy
  [YEZ] 2024-04-11: 1,584 words → YEZ_2024-04-11_word_idx.npy
  [YEZ] 2024-04-12: 7,996 words → YEZ_2024-04-12_word_idx.npy
  [YEZ] 2024-04-14: 18,298 words → YEZ_2024-04-14_word_idx.npy
  [YEZ] 2024-04-15: 14,595 words → YEZ_2024-04-15_word_idx.npy
  [YFA] 2024-04-23: 18,729 words → YFA_2024-04-23_word_idx.npy
  [YFA] 2024-04-24: 58,808 words → YFA_2024-04-24_word_idx.npy
  [YFA] 2024-04-25: 48,473 words → YFA_2024-04-25_word_idx.npy
  [YFA] 2024-04-26: 36,215 words → YFA_2024-04-26_word_idx.npy
  [YFA] 2024-04-27: 66,950 words → YFA_2024-04-27_word_idx.npy
  [YFA] 2024-04-28: 43,939 words → YFA_2024-04-28_word_idx.npy
  [YFA] 2024-04-29: 54,695 words → YFA_2024-04-29_word_idx.npy
  [YFA]

### 4. Path Resolution & Consensus-Params Helpers

In [4]:
def first_existing(paths):
    for p in paths:
        if p is not None and Path(p).exists():
            return Path(p)
    return None


def resolve_patient_neural_inputs(patient: str) -> dict:
    root = VAD_ROOT / patient
    cluster_override = CLUSTER_PREDS_OVERRIDE.get(patient)
    frs_override     = FRS_PATH_OVERRIDE.get(patient)

    new_clusters = first_existing([root / 'embeddings' / 'semantic_cluster_predictions.npy'])
    new_frs = first_existing([
        root / 'neural_embeddings' / 'word_frs.npy',
        root / 'all_convo_recording' / 'word_avg_frs.npy',
    ])
    legacy_frs = first_existing([LEGACY_ROOT / patient / 'all_convo_recording' / 'word_avg_frs.npy'])

    if cluster_override is not None or frs_override is not None:
        cluster_preds_path = first_existing([cluster_override, new_clusters])
        frs_path           = first_existing([frs_override, new_frs, legacy_frs])
    elif new_clusters is not None and new_frs is not None:
        cluster_preds_path = new_clusters
        frs_path           = new_frs
    else:
        cluster_preds_path = None
        frs_path           = None

    return {'cluster_preds_path': cluster_preds_path, 'frs_path': frs_path}


def patient_consensus_path(patient: str) -> 'Path | None':
    """Return existing consensus params from the base patient-level run, if present."""
    p = VAD_ROOT / patient / 'standard_decoding_analysis' / PATIENT_CONSENSUS_RUN / 'consensus_best_params.json'
    return p if p.exists() else None


def choose_consensus_params(param_options: list) -> dict:
    params = dict(param_options[0])
    params['max_depth']        = int(np.min([p['max_depth'] for p in param_options]))
    params['min_child_weight'] = int(np.max([p['min_child_weight'] for p in param_options]))
    params['gamma']            = float(np.max([p['gamma'] for p in param_options]))
    params['reg_lambda']       = float(np.max([p['reg_lambda'] for p in param_options]))
    params['reg_alpha']        = float(np.max([p['reg_alpha'] for p in param_options]))
    params['subsample']        = float(np.median([p['subsample'] for p in param_options]))
    params['colsample_bytree'] = float(np.median([p['colsample_bytree'] for p in param_options]))
    params['learning_rate']    = float(np.median([p['learning_rate'] for p in param_options]))
    params['n_estimators']     = int(np.median([p['n_estimators'] for p in param_options]))
    params['tree_method'] = 'hist'
    params['device'] = 'cpu'
    params.pop('predictor', None)
    params.pop('use_label_encoder', None)
    return params

### 5. Build Status Table

In [5]:
def resample_paths(out_root: Path, r: int) -> dict:
    return {
        'pkl':              out_root / f'scat_sampled_resample_{r}_.pkl',
        'summary_json':     out_root / f'summary_resample_{r}.json',
        'best_params_json': out_root / f'best_params_resample_{r}.json',
        'success':          out_root / f'resample_{r}_SUCCESS',
        'error':            out_root / f'resample_{r}_error.txt',
    }


def build_status_df() -> pd.DataFrame:
    rows = []
    for patient in PATIENTS:
        neural = resolve_patient_neural_inputs(patient)
        day_idx_map = patient_day_idx_paths.get(patient, {})
        borrowed_consensus = patient_consensus_path(patient) if REUSE_PATIENT_CONSENSUS else None

        for date_str, word_idx_path in day_idx_map.items():
            out_root = VAD_ROOT / patient / 'standard_decoding_analysis' / BASE_RUN_NAME / date_str
            out_root.mkdir(parents=True, exist_ok=True)
            consensus_json = out_root / 'consensus_best_params.json'
            has_inputs = (neural['cluster_preds_path'] is not None
                         and neural['frs_path'] is not None)

            # When reusing patient consensus, copy it into the day output dir if not there yet
            if REUSE_PATIENT_CONSENSUS and borrowed_consensus is not None and not consensus_json.exists():
                consensus_json.write_text(borrowed_consensus.read_text())

            consensus_ready = consensus_json.exists()

            # Which resamples to run:
            # - REUSE_PATIENT_CONSENSUS=True  → all 20 go directly to fixed-params stage
            # - REUSE_PATIENT_CONSENSUS=False → normal 3+17 flow
            if REUSE_PATIENT_CONSENSUS:
                effective_first_stage  = []
                effective_second_stage = list(range(N_RESAMPLES))
            else:
                effective_first_stage  = FIRST_STAGE
                effective_second_stage = SECOND_STAGE

            first_three_done = all(
                (out_root / f'resample_{r}_SUCCESS').exists()
                for r in effective_first_stage
            ) if effective_first_stage else True

            for r in range(N_RESAMPLES):
                paths = resample_paths(out_root, r)
                rows.append({
                    'patient':             patient,
                    'date':                date_str,
                    'resample_idx':        r,
                    'stage':               'hyperparam' if r in effective_first_stage else 'fixed',
                    'seed':                r * SEED_STRIDE,
                    'cluster_preds_path':  neural['cluster_preds_path'],
                    'frs_path':            neural['frs_path'],
                    'word_idx_path':       word_idx_path,
                    'has_inputs':          has_inputs,
                    'out_root':            out_root,
                    'consensus_json':      consensus_json,
                    'consensus_ready':     consensus_ready,
                    'first_three_done':    first_three_done,
                    'pkl_path':            paths['pkl'],
                    'summary_json':        paths['summary_json'],
                    'best_params_json':    paths['best_params_json'],
                    'success_path':        paths['success'],
                    'error_path':          paths['error'],
                    'has_pkl':             paths['pkl'].exists(),
                    'has_success':         paths['success'].exists(),
                    'has_error':           paths['error'].exists(),
                    'ready_phase1':        (has_inputs and r in effective_first_stage
                                          and not paths['success'].exists()),
                    'ready_phase2':        (has_inputs and r in effective_second_stage
                                          and first_three_done and consensus_ready
                                          and not paths['success'].exists()),
                })
    return pd.DataFrame(rows).sort_values(['patient', 'date', 'resample_idx']).reset_index(drop=True)


status_df = build_status_df()
print(f'total rows (patient × day × resample): {len(status_df)}')
print(f'has inputs:      {int(status_df["has_inputs"].sum())}')
print(f'consensus ready: {int(status_df["consensus_ready"].sum())}')
print(f'completed:       {int(status_df["has_success"].sum())}')
print(f'errors:          {int(status_df["has_error"].sum())}')
print(f'ready_phase1:    {int(status_df["ready_phase1"].sum())}')
print(f'ready_phase2:    {int(status_df["ready_phase2"].sum())}')
status_df[['patient','date','resample_idx','stage','has_inputs','consensus_ready',
           'has_success','has_error','ready_phase1','ready_phase2']].head(40)

total rows (patient × day × resample): 2940
has inputs:      2940
consensus ready: 2940
completed:       2900
errors:          6
ready_phase1:    6
ready_phase2:    0


,patient,date,resample_idx,stage,has_inputs,consensus_ready,has_success,has_error,ready_phase1,ready_phase2
0,YEY,2024-04-01,0,hyperparam,True,True,True,False,False,False
1,YEY,2024-04-01,1,hyperparam,True,True,True,False,False,False
2,YEY,2024-04-01,2,hyperparam,True,True,True,False,False,False
3,YEY,2024-04-01,3,fixed,True,True,True,False,False,False
4,YEY,2024-04-01,4,fixed,True,True,True,False,False,False
5,YEY,2024-04-01,5,fixed,True,True,True,False,False,False
6,YEY,2024-04-01,6,fixed,True,True,True,False,False,False
7,YEY,2024-04-01,7,fixed,True,True,True,False,False,False
8,YEY,2024-04-01,8,fixed,True,True,True,False,False,False
9,YEY,2024-04-01,9,fixed,True,True,True,False,False,False


### 6. Patient × Day Input Summary

In [6]:
pair_df = (
    status_df.drop_duplicates(subset=['patient', 'date'])
    [['patient', 'date', 'has_inputs', 'consensus_ready', 'cluster_preds_path', 'frs_path', 'word_idx_path']]
    .sort_values(['patient', 'date'])
    .reset_index(drop=True)
)
print(f'{len(pair_df)} (patient, day) pairs')
display(pair_df)

147 (patient, day) pairs


,patient,date,has_inputs,consensus_ready,cluster_preds_path,frs_path,word_idx_path
0,YEY,2024-04-01,True,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
1,YEY,2024-04-02,True,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
2,YEZ,2024-04-09,True,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
3,YEZ,2024-04-10,True,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
4,YEZ,2024-04-11,True,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
5,YEZ,2024-04-12,True,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
6,YEZ,2024-04-14,True,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
7,YEZ,2024-04-15,True,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
8,YFA,2024-04-23,True,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
9,YFA,2024-04-24,True,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...


### 7a. Submit Phase-1 Jobs  
*(Only runs if `REUSE_PATIENT_CONSENSUS = False`; otherwise `ready_phase1` is always empty.)*

In [7]:
DRY_RUN = False

submitted = []
failed    = []

for _, row in status_df[status_df['ready_phase1']].iterrows():
    patient  = row['patient']
    date_str = row['date']
    r        = int(row['resample_idx'])
    logs_dir    = row['out_root'] / 'slurm_logs'
    scripts_dir = row['out_root'] / 'slurm_scripts'
    logs_dir.mkdir(parents=True, exist_ok=True)
    scripts_dir.mkdir(parents=True, exist_ok=True)

    word_idx_arg = f'--word-idx-path "{row["word_idx_path"]}"' if row['word_idx_path'] else ''
    sbatch_text = f"""#!/bin/bash
#SBATCH --job-name=scat_pd_{patient}_{date_str}_{r}
#SBATCH --partition={PARTITION}
#SBATCH --qos={QOS}
#SBATCH --cpus-per-task={CPUS}
#SBATCH --mem={MEM}
#SBATCH --time={TIME}
#SBATCH --output={logs_dir}/{patient}_{date_str}_r{r}_%j.out
#SBATCH --error={logs_dir}/{patient}_{date_str}_r{r}_%j.err

set -eo pipefail
echo "HOSTNAME: $(hostname)"
echo "START: $(date)"
echo "PATIENT: {patient}"
echo "DATE: {date_str}"
echo "RESAMPLE: {r}"

{PYTHON_BIN} {WORKER_PATH} \\
  --patient {patient} \\
  --resample-idx {r} \\
  --cluster-preds-path "{row['cluster_preds_path']}" \\
  --frs-path "{row['frs_path']}" \\
  --out-dir "{row['out_root']}" \\
  {word_idx_arg} \\
  --test-size {TEST_SIZE} \\
  --n-iter {SEARCH_ITERS} \\
  --cv-splits {CV_SPLITS} \\
  --n-shuffles {N_SHUFFLES} \\
  --seed-stride {SEED_STRIDE} \\
  --n-jobs {CPUS * 2} \\
  --scoring {SCORING} \\
  --perm-metric {PERM_METRIC}

echo "END: $(date)"
"""
    sbatch_path = scripts_dir / f'{patient}_{date_str}_resample_{r}.sbatch'
    sbatch_path.write_text(sbatch_text)

    if DRY_RUN:
        submitted.append((patient, date_str, r, 'dry-run'))
        print(f'dry-run: {patient} {date_str} r={r}')
    else:
        try:
            res = subprocess.run(['sbatch', str(sbatch_path)], capture_output=True, text=True, check=True)
            submitted.append((patient, date_str, r, res.stdout.strip()))
            print(f'submitted: {patient} {date_str} r={r} -> {res.stdout.strip()}')
        except subprocess.CalledProcessError as exc:
            failed.append((patient, date_str, r, exc.stderr.strip()))
            print(f'FAILED: {patient} {date_str} r={r}\n{exc.stderr}')

print(f'phase-1 submitted={len(submitted)}, failed={len(failed)}')

submitted: YFG 2024-09-23 r=0 -> Submitted batch job 274401
submitted: YFG 2024-09-23 r=1 -> Submitted batch job 274402
submitted: YFG 2024-09-23 r=2 -> Submitted batch job 274403
submitted: YFR 2025-06-30 r=0 -> Submitted batch job 274404
submitted: YFR 2025-06-30 r=1 -> Submitted batch job 274405
submitted: YFR 2025-06-30 r=2 -> Submitted batch job 274406
phase-1 submitted=6, failed=0


### 7b. Build Per-Day Consensus Params  
*(Only needed if `REUSE_PATIENT_CONSENSUS = False`; otherwise consensus was copied in the status step.)*

In [8]:
if not REUSE_PATIENT_CONSENSUS:
    built   = []
    blocked = []
    for (patient, date_str), grp in status_df.groupby(['patient', 'date']):
        out_root       = grp.iloc[0]['out_root']
        consensus_json = grp.iloc[0]['consensus_json']
        if consensus_json.exists():
            print(f'consensus already exists: {patient} {date_str}')
            continue
        first_three = grp[grp['resample_idx'].isin(FIRST_STAGE)]
        if not bool(first_three['has_success'].all()):
            blocked.append((patient, date_str))
            continue
        param_options = []
        for r in FIRST_STAGE:
            path = out_root / f'best_params_resample_{r}.json'
            param_options.append(json.loads(path.read_text()))
        consensus = choose_consensus_params(param_options)
        consensus_json.write_text(json.dumps(consensus, indent=2))
        built.append((patient, date_str))
        print(f'built consensus: {patient} {date_str}')
    print('built:', len(built), '  blocked:', len(blocked))
else:
    print('REUSE_PATIENT_CONSENSUS=True — consensus already copied during status build.')

consensus already exists: YEY 2024-04-01
consensus already exists: YEY 2024-04-02
consensus already exists: YEZ 2024-04-09
consensus already exists: YEZ 2024-04-10
consensus already exists: YEZ 2024-04-11
consensus already exists: YEZ 2024-04-12
consensus already exists: YEZ 2024-04-14
consensus already exists: YEZ 2024-04-15
consensus already exists: YFA 2024-04-23
consensus already exists: YFA 2024-04-24
consensus already exists: YFA 2024-04-25
consensus already exists: YFA 2024-04-26
consensus already exists: YFA 2024-04-27
consensus already exists: YFA 2024-04-28
consensus already exists: YFA 2024-04-29
consensus already exists: YFA 2024-04-30
consensus already exists: YFA 2024-05-01
consensus already exists: YFB 2024-05-02
consensus already exists: YFB 2024-05-03
consensus already exists: YFB 2024-05-04
consensus already exists: YFB 2024-05-05
consensus already exists: YFB 2024-05-06
consensus already exists: YFB 2024-05-07
consensus already exists: YFB 2024-05-08
consensus alread

### 7c. Submit Phase-2 (Fixed-Params) Jobs

In [9]:
DRY_RUN = False

# Refresh status after any consensus files were written
status_df = build_status_df()

submitted = []
failed    = []

for _, row in status_df[status_df['ready_phase2']].iterrows():
    patient  = row['patient']
    date_str = row['date']
    r        = int(row['resample_idx'])
    logs_dir    = row['out_root'] / 'slurm_logs'
    scripts_dir = row['out_root'] / 'slurm_scripts'
    logs_dir.mkdir(parents=True, exist_ok=True)
    scripts_dir.mkdir(parents=True, exist_ok=True)

    word_idx_arg = f'--word-idx-path "{row["word_idx_path"]}"' if row['word_idx_path'] else ''
    sbatch_text = f"""#!/bin/bash
#SBATCH --job-name=scat_pd_{patient}_{date_str}_{r}
#SBATCH --partition={PARTITION}
#SBATCH --qos={QOS}
#SBATCH --cpus-per-task={CPUS}
#SBATCH --mem={MEM}
#SBATCH --time={TIME}
#SBATCH --output={logs_dir}/{patient}_{date_str}_r{r}_%j.out
#SBATCH --error={logs_dir}/{patient}_{date_str}_r{r}_%j.err

set -eo pipefail
echo "HOSTNAME: $(hostname)"
echo "START: $(date)"
echo "PATIENT: {patient}"
echo "DATE: {date_str}"
echo "RESAMPLE: {r}"

{PYTHON_BIN} {WORKER_PATH} \\
  --patient {patient} \\
  --resample-idx {r} \\
  --cluster-preds-path "{row['cluster_preds_path']}" \\
  --frs-path "{row['frs_path']}" \\
  --out-dir "{row['out_root']}" \\
  --params-json "{row['consensus_json']}" \\
  {word_idx_arg} \\
  --test-size {TEST_SIZE} \\
  --n-iter {SEARCH_ITERS} \\
  --cv-splits {CV_SPLITS} \\
  --n-shuffles {N_SHUFFLES} \\
  --seed-stride {SEED_STRIDE} \\
  --n-jobs {CPUS * 2} \\
  --scoring {SCORING} \\
  --perm-metric {PERM_METRIC}

echo "END: $(date)"
"""
    sbatch_path = scripts_dir / f'{patient}_{date_str}_resample_{r}.sbatch'
    sbatch_path.write_text(sbatch_text)

    if DRY_RUN:
        submitted.append((patient, date_str, r, 'dry-run'))
        print(f'dry-run: {patient} {date_str} r={r}')
    else:
        try:
            res = subprocess.run(['sbatch', str(sbatch_path)], capture_output=True, text=True, check=True)
            submitted.append((patient, date_str, r, res.stdout.strip()))
            print(f'submitted: {patient} {date_str} r={r} -> {res.stdout.strip()}')
        except subprocess.CalledProcessError as exc:
            failed.append((patient, date_str, r, exc.stderr.strip()))
            print(f'FAILED: {patient} {date_str} r={r}\n{exc.stderr}')

print(f'phase-2 submitted={len(submitted)}, failed={len(failed)}')

phase-2 submitted=0, failed=0


### 8. Check Status

In [10]:
status_df = build_status_df()
pair_progress = (
    status_df.groupby(['patient', 'date'])
    .agg(
        total=('resample_idx', 'count'),
        done=('has_success', 'sum'),
        errors=('has_error', 'sum'),
    )
    .reset_index()
)
pair_progress['pct'] = (pair_progress['done'] / pair_progress['total'] * 100).round(0).astype(int)
print(f'Overall: {int(status_df["has_success"].sum())} / {len(status_df)} resamples done')
display(pair_progress)

Overall: 2900 / 2940 resamples done


,patient,date,total,done,errors,pct
0,YEY,2024-04-01,20,20,0,100
1,YEY,2024-04-02,20,20,0,100
2,YEZ,2024-04-09,20,20,0,100
3,YEZ,2024-04-10,20,20,0,100
4,YEZ,2024-04-11,20,20,0,100
5,YEZ,2024-04-12,20,20,0,100
6,YEZ,2024-04-14,20,20,0,100
7,YEZ,2024-04-15,20,20,0,100
8,YFA,2024-04-23,20,20,0,100
9,YFA,2024-04-24,20,20,0,100


### 9. Inspect Errors

In [11]:
error_rows = status_df[status_df['has_error']].copy()
print(f'resamples with error files: {len(error_rows)}')
for _, row in error_rows.head(10).iterrows():
    print('=' * 100)
    print(row['patient'], row['date'], f'r={row["resample_idx"]}')
    print(row['error_path'].read_text()[:4000])

resamples with error files: 6
YFG 2024-09-23 r=0
Traceback (most recent call last):
  File "/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/standard_decoding_analysis/scat_classifier_worker.py", line 377, in main
    model, results = run_one_resample(
                     ^^^^^^^^^^^^^^^^^
  File "/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/standard_decoding_analysis/scat_classifier_worker.py", line 152, in run_one_resample
    Xtr, Xte, ytr, yte = train_test_split(
                         ^^^^^^^^^^^^^^^^^
  File "/scratch/tahaismail424/miniforge3/envs/speech_247_env/lib/python3.11/site-packages/sklearn/utils/_param_validation.py", line 218, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/scratch/tahaismail424/miniforge3/envs/speech_247_env/lib/python3.11/site-packages/sklearn/model_selection/_split.py", line 2945, in train_test_split
    train, test = next(cv.split(X=arrays[0], y=stratify))
                  ^^^^^^^^^^^^^^^^^^^

### 10. Aggregate Results Per (Patient, Day)

In [12]:
for (patient, date_str), grp in status_df.groupby(['patient', 'date']):
    if not bool(grp['has_success'].all()):
        print(f'skipping {patient} {date_str}: not all resamples finished')
        continue

    out_root     = grp.iloc[0]['out_root']
    summary_rows = []
    aggregate    = {}
    for r in range(N_RESAMPLES):
        with open(out_root / f'summary_resample_{r}.json') as f:
            summary_rows.append(json.load(f))
        with open(out_root / f'scat_sampled_resample_{r}_.pkl', 'rb') as f:
            aggregate[f'resample_{r}'] = pickle.load(f)

    summary_df = pd.DataFrame(summary_rows).sort_values('resample_idx').reset_index(drop=True)
    agg_csv = out_root / 'scat_sampled_all_resamples.csv'
    agg_pkl = out_root / 'scat_sampled_all_resamples.pkl'
    summary_df.to_csv(agg_csv, index=False)
    with open(agg_pkl, 'wb') as f:
        pickle.dump(aggregate, f)
    print(f'aggregated {patient} {date_str} -> {agg_csv}')
    display(summary_df)

aggregated YEY 2024-04-01 -> /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YEY/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered_per_day/2024-04-01/scat_sampled_all_resamples.csv


,patient,resample_idx,seed,n_sampled_raw,n_sampled_final,n_test,acc,balanced_accuracy,f1_macro,f1_micro,auc_macro_ovr,baseline_proportional_acc,baseline_majority_acc,binomial_p_vs_proportional,perm_metric,observed_perm_metric,perm_pvalue,cv_best_score,perm_null
0,YEY,0,0,3871,3871,775,0.153548,0.125827,0.124919,0.153548,0.545338,0.118375,0.134194,2.004897e-03,f1_macro,0.124919,0.019608,0.136489,shuffle_training_X_only
1,YEY,1,42,3871,3871,775,0.149677,0.119451,0.116253,0.149677,0.563953,0.118375,0.134194,5.138033e-03,f1_macro,0.116253,0.019608,0.142778,shuffle_training_X_only
2,YEY,2,84,3871,3871,775,0.172903,0.138114,0.135387,0.172903,0.564575,0.118375,0.134194,5.406343e-06,f1_macro,0.135387,0.019608,0.142540,shuffle_training_X_only
3,YEY,3,126,3871,3871,775,0.179355,0.146221,0.144660,0.179355,0.576744,0.118375,0.134194,4.867366e-07,f1_macro,0.144660,0.019608,NaN,shuffle_training_X_only
4,YEY,4,168,3871,3871,775,0.176774,0.140854,0.138499,0.176774,0.557263,0.118375,0.134194,1.308017e-06,f1_macro,0.138499,0.019608,NaN,shuffle_training_X_only
5,YEY,5,210,3871,3871,775,0.197419,0.158467,0.155698,0.197419,0.560260,0.118375,0.134194,1.903913e-10,f1_macro,0.155698,0.019608,NaN,shuffle_training_X_only
6,YEY,6,252,3871,3871,775,0.172903,0.141460,0.138985,0.172903,0.569691,0.118375,0.134194,5.406343e-06,f1_macro,0.138985,0.019608,NaN,shuffle_training_X_only
7,YEY,7,294,3871,3871,775,0.152258,0.120255,0.116493,0.152258,0.559790,0.118375,0.134194,2.768766e-03,f1_macro,0.116493,0.039216,NaN,shuffle_training_X_only
8,YEY,8,336,3871,3871,775,0.161290,0.127814,0.123680,0.161290,0.559582,0.118375,0.134194,2.390449e-04,f1_macro,0.123680,0.019608,NaN,shuffle_training_X_only
9,YEY,9,378,3871,3871,775,0.162581,0.132700,0.130432,0.162581,0.592257,0.118375,0.134194,1.625253e-04,f1_macro,0.130432,0.019608,NaN,shuffle_training_X_only


aggregated YEY 2024-04-02 -> /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YEY/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered_per_day/2024-04-02/scat_sampled_all_resamples.csv


,patient,resample_idx,seed,n_sampled_raw,n_sampled_final,n_test,acc,balanced_accuracy,f1_macro,f1_micro,auc_macro_ovr,baseline_proportional_acc,baseline_majority_acc,binomial_p_vs_proportional,perm_metric,observed_perm_metric,perm_pvalue,cv_best_score,perm_null
0,YEY,0,0,4892,4892,979,0.163432,0.123227,0.113592,0.163432,0.542375,0.119912,0.137896,3.618538e-05,f1_macro,0.113592,0.019608,0.12575,shuffle_training_X_only
1,YEY,1,42,4892,4892,979,0.157303,0.122771,0.120563,0.157303,0.533602,0.119912,0.137896,3.076525e-04,f1_macro,0.120563,0.019608,0.12883,shuffle_training_X_only
2,YEY,2,84,4892,4892,979,0.194076,0.148285,0.140803,0.194076,0.548925,0.119912,0.137896,1.984121e-11,f1_macro,0.140803,0.019608,0.12931,shuffle_training_X_only
3,YEY,3,126,4892,4892,979,0.171604,0.129152,0.121098,0.171604,0.535959,0.119912,0.137896,1.405157e-06,f1_macro,0.121098,0.019608,NaN,shuffle_training_X_only
4,YEY,4,168,4892,4892,979,0.194076,0.147441,0.138261,0.194076,0.566975,0.119912,0.137896,1.984121e-11,f1_macro,0.138261,0.019608,NaN,shuffle_training_X_only
5,YEY,5,210,4892,4892,979,0.162411,0.124818,0.120223,0.162411,0.543717,0.119912,0.137896,5.262352e-05,f1_macro,0.120223,0.039216,NaN,shuffle_training_X_only
6,YEY,6,252,4892,4892,979,0.153218,0.117922,0.113949,0.153218,0.539144,0.119912,0.137896,1.110232e-03,f1_macro,0.113949,0.058824,NaN,shuffle_training_X_only
7,YEY,7,294,4892,4892,979,0.165475,0.127258,0.121923,0.165475,0.556789,0.119912,0.137896,1.675114e-05,f1_macro,0.121923,0.019608,NaN,shuffle_training_X_only
8,YEY,8,336,4892,4892,979,0.167518,0.126743,0.118823,0.167518,0.556236,0.119912,0.137896,7.539806e-06,f1_macro,0.118823,0.019608,NaN,shuffle_training_X_only
9,YEY,9,378,4892,4892,979,0.167518,0.129796,0.124739,0.167518,0.552660,0.119912,0.137896,7.539806e-06,f1_macro,0.124739,0.019608,NaN,shuffle_training_X_only


aggregated YEZ 2024-04-09 -> /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YEZ/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered_per_day/2024-04-09/scat_sampled_all_resamples.csv


...

,patient,resample_idx,seed,n_sampled_raw,n_sampled_final,n_test,acc,balanced_accuracy,f1_macro,f1_micro,auc_macro_ovr,baseline_proportional_acc,baseline_majority_acc,binomial_p_vs_proportional,perm_metric,observed_perm_metric,perm_pvalue,cv_best_score,perm_null
0,YFV,0,0,2860,2860,572,0.187063,0.144961,0.132776,0.187063,0.593522,0.11717,0.134615,8.087763e-07,f1_macro,0.132776,0.019608,0.144571,shuffle_training_X_only
1,YFV,1,42,2860,2860,572,0.166084,0.134241,0.129104,0.166084,0.569852,0.11717,0.134615,3.282484e-04,f1_macro,0.129104,0.019608,0.141671,shuffle_training_X_only
2,YFV,2,84,2860,2860,572,0.174825,0.138629,0.132530,0.174825,0.584564,0.11717,0.134615,3.281510e-05,f1_macro,0.132530,0.019608,0.126898,shuffle_training_X_only
3,YFV,3,126,2860,2860,572,0.178322,0.143606,0.136649,0.178322,0.566946,0.11717,0.134615,1.205122e-05,f1_macro,0.136649,0.019608,NaN,shuffle_training_X_only
4,YFV,4,168,2860,2860,572,0.190559,0.155191,0.149921,0.190559,0.584190,0.11717,0.134615,2.540237e-07,f1_macro,0.149921,0.019608,NaN,shuffle_training_X_only
5,YFV,5,210,2860,2860,572,0.162587,0.129060,0.123835,0.162587,0.563400,0.11717,0.134615,7.598228e-04,f1_macro,0.123835,0.019608,NaN,shuffle_training_X_only
6,YFV,6,252,2860,2860,572,0.153846,0.123310,0.120487,0.153846,0.567508,0.11717,0.134615,5.028310e-03,f1_macro,0.120487,0.019608,NaN,shuffle_training_X_only
7,YFV,7,294,2860,2860,572,0.150350,0.117191,0.110224,0.150350,0.554755,0.11717,0.134615,9.840931e-03,f1_macro,0.110224,0.098039,NaN,shuffle_training_X_only
8,YFV,8,336,2860,2860,572,0.166084,0.133603,0.130120,0.166084,0.591751,0.11717,0.134615,3.282484e-04,f1_macro,0.130120,0.019608,NaN,shuffle_training_X_only
9,YFV,9,378,2860,2860,572,0.173077,0.142791,0.138579,0.173077,0.588734,0.11717,0.134615,5.322924e-05,f1_macro,0.138579,0.019608,NaN,shuffle_training_X_only


aggregated YFV 2026-02-19 -> /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFV/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered_per_day/2026-02-19/scat_sampled_all_resamples.csv


,patient,resample_idx,seed,n_sampled_raw,n_sampled_final,n_test,acc,balanced_accuracy,f1_macro,f1_micro,auc_macro_ovr,baseline_proportional_acc,baseline_majority_acc,binomial_p_vs_proportional,perm_metric,observed_perm_metric,perm_pvalue,cv_best_score,perm_null
0,YFV,0,0,7284,5205,1041,0.198847,0.152796,0.152990,0.198847,0.569169,0.127943,0.155620,8.827948e-11,f1_macro,0.152990,0.019608,0.142922,shuffle_training_X_only
1,YFV,1,42,7284,5194,1039,0.216554,0.152781,0.144094,0.216554,0.592658,0.127937,0.155919,1.801120e-15,f1_macro,0.144094,0.019608,0.140041,shuffle_training_X_only
2,YFV,2,84,7284,5204,1041,0.196926,0.148968,0.147085,0.196926,0.581760,0.128079,0.157541,2.786125e-10,f1_macro,0.147085,0.019608,0.145257,shuffle_training_X_only
3,YFV,3,126,7284,5238,1048,0.202290,0.151249,0.149898,0.202290,0.605492,0.128161,0.157443,1.246806e-11,f1_macro,0.149898,0.019608,NaN,shuffle_training_X_only
4,YFV,4,168,7284,5229,1046,0.196941,0.143188,0.135343,0.196941,0.568043,0.128241,0.156788,2.792482e-10,f1_macro,0.135343,0.019608,NaN,shuffle_training_X_only
5,YFV,5,210,7284,5204,1041,0.194044,0.145505,0.141956,0.194044,0.586444,0.128044,0.158501,1.273484e-09,f1_macro,0.141956,0.019608,NaN,shuffle_training_X_only
6,YFV,6,252,7284,5186,1038,0.202312,0.150608,0.146291,0.202312,0.594579,0.128047,0.158960,1.415057e-11,f1_macro,0.146291,0.019608,NaN,shuffle_training_X_only
7,YFV,7,294,7284,5201,1041,0.211335,0.155372,0.151019,0.211335,0.581828,0.127943,0.155620,5.105309e-14,f1_macro,0.151019,0.019608,NaN,shuffle_training_X_only
8,YFV,8,336,7284,5207,1042,0.203455,0.152391,0.149798,0.203455,0.581363,0.128061,0.157390,6.721539e-12,f1_macro,0.149798,0.019608,NaN,shuffle_training_X_only
9,YFV,9,378,7284,5194,1039,0.201155,0.145787,0.141422,0.201155,0.576156,0.127902,0.154957,2.431745e-11,f1_macro,0.141422,0.019608,NaN,shuffle_training_X_only


aggregated YFV 2026-02-20 -> /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFV/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered_per_day/2026-02-20/scat_sampled_all_resamples.csv


,patient,resample_idx,seed,n_sampled_raw,n_sampled_final,n_test,acc,balanced_accuracy,f1_macro,f1_micro,auc_macro_ovr,baseline_proportional_acc,baseline_majority_acc,binomial_p_vs_proportional,perm_metric,observed_perm_metric,perm_pvalue,cv_best_score,perm_null
0,YFV,0,0,9234,7830,1566,0.236271,0.170019,0.155493,0.236271,0.630881,0.123723,0.150064,1.221263e-34,f1_macro,0.155493,0.019608,0.163232,shuffle_training_X_only
1,YFV,1,42,9234,7824,1565,0.222364,0.162874,0.155534,0.222364,0.644866,0.123604,0.149521,1.311618e-27,f1_macro,0.155534,0.019608,0.164978,shuffle_training_X_only
2,YFV,2,84,9234,7809,1562,0.227273,0.167967,0.159414,0.227273,0.619159,0.123491,0.150448,4.761547e-30,f1_macro,0.159414,0.019608,0.157512,shuffle_training_X_only
3,YFV,3,126,9234,7788,1558,0.208601,0.154564,0.147870,0.208601,0.627365,0.123370,0.148909,2.289500e-21,f1_macro,0.147870,0.019608,NaN,shuffle_training_X_only
4,YFV,4,168,9234,7789,1558,0.222080,0.162485,0.153212,0.222080,0.623442,0.123483,0.147625,1.970896e-27,f1_macro,0.153212,0.019608,NaN,shuffle_training_X_only
5,YFV,5,210,9234,7850,1570,0.214650,0.162623,0.158896,0.214650,0.622272,0.123861,0.150955,6.426299e-24,f1_macro,0.158896,0.019608,NaN,shuffle_training_X_only
6,YFV,6,252,9234,7798,1560,0.207051,0.154325,0.148137,0.207051,0.636376,0.123565,0.148077,1.250726e-20,f1_macro,0.148137,0.019608,NaN,shuffle_training_X_only
7,YFV,7,294,9234,7816,1564,0.210997,0.161787,0.157753,0.210997,0.627000,0.123540,0.148977,2.169185e-22,f1_macro,0.157753,0.019608,NaN,shuffle_training_X_only
8,YFV,8,336,9234,7792,1559,0.234766,0.174756,0.165791,0.234766,0.624739,0.123564,0.148172,8.330592e-34,f1_macro,0.165791,0.019608,NaN,shuffle_training_X_only
9,YFV,9,378,9234,7827,1566,0.211367,0.159576,0.155905,0.211367,0.618965,0.123608,0.150702,1.533986e-22,f1_macro,0.155905,0.019608,NaN,shuffle_training_X_only
